# A — Chuẩn bị dữ liệu bốn loài, bổ sung training D5

**Mục tiêu:** giữ dữ liệu D1–D4 của Ae. aegypti, Ae. albopictus, An. dirus và An. minimus; bổ sung có kiểm soát **Ae. albopictus từ training D5**. Giữ toàn bộ window hợp lệ của các loài khác, chưa ép cân bằng số lượng.

**Đầu ra:** hai manifest `biodcase_four_species_clips.csv`, `biodcase_four_species_windows.csv`, bảng/biểu đồ và `results/A/preparation/report.html`. Thiết lập trực tiếp trong cell đầu, không đọc hoặc sửa `config.json`. Đọc WAV gốc từ Downloads; chỉ xuất chỉ mục window, không xuất WAV nhỏ hàng loạt.

Chạy **Run All** bằng Python `.venv`. Cần kết quả inventory và bảng đối chiếu HumBugDB từ notebook 02. Notebook đọc lại toàn bộ audio được chọn để kiểm tra chất lượng và trùng lặp.

**Giới hạn:** chưa xác minh cá thể/nguồn gốc độc lập cho mọi clip; `name` HumBugDB chỉ là dấu vết nguồn. D5 chỉ bổ sung cho một loài nên có nguy cơ gắn điều kiện thu với nhãn. Validation/test giữ D1–D4, không bổ sung D5 và không điều chỉnh cho cân bằng. Protocol A_v1 đã chốt cho thí nghiệm chuyển domain theo từng loài; chưa đủ bằng chứng cho đánh giá độc lập cá thể.

Nguồn: [BioDCASE Task 5](https://biodcase.github.io/challenge2026/task5), metadata tải kèm và kết quả Step A local.

**Bản chốt A_v1:** `reference_split` giữ cách chia tham chiếu sau làm sạch; `split` là cách chia nghiên cứu theo nhóm nguồn đã biết và các đoạn trùng phát hiện. D5 vẫn chỉ training. Test nghiên cứu giữ domain chưa thấy theo từng loài. Validation không còn chia ngẫu nhiên clip cùng `name`; riêng Ae. albopictus dùng toàn bộ nguồn D1 làm validation, train D5 và test D2.

Chỉ bảo đảm không giao nhau ở những nhóm đã biết/phát hiện; không gọi đây là đánh giá độc lập cá thể cho các nguồn chưa có metadata. Domain test được giữ riêng theo từng loài, không phải một domain chưa xuất hiện ở toàn bộ training. A_v1 là protocol thăm dò duy nhất được kích hoạt; chưa tạo thêm một phép chia ngẫu nhiên để coi là benchmark độc lập nguồn.

**Phạm vi cảm biến:** metadata hiện tại chưa đủ để xác minh loại cảm biến cho mọi domain, đặc biệt D5. A_v1 là benchmark thăm dò về phân loại tín hiệu ngắn; chưa được gọi là bộ microphone-only đã xác minh. Đuôi `.wav`, sample rate và độ dài không chứng minh tín hiệu được thu bằng microphone.

## 1. Thiết lập trực tiếp
Sửa đường dẫn và các biến ở đây. `D5_ALBOPICTUS_CLIPS=1374` là lượng bổ sung khởi đầu, có thể điều chỉnh trong thí nghiệm sau; không phải ngưỡng đã tối ưu. Chọn ngẫu nhiên không hoàn lại bằng seed cố định từ **training D5**, sau khi sắp xếp ID để tái lập.

Mono 8 kHz, window 300 ms, hop 150 ms. Giữ toàn bộ window hợp lệ; sampling, weighted loss và cấu hình cân bằng tuyệt đối là những đối chứng ở phần B–C, không áp dụng ngay trong bước chuẩn bị.

Các ngưỡng tìm gần trùng chỉ dùng để phát hiện khả năng rò rỉ, không tối ưu bằng kết quả model. Đoạn quá nhỏ (< −100 dBFS theo độ lệch chuẩn) không dùng xác nhận cùng nguồn. Bản ghi nhỏ tiếng vẫn được giữ.

In [ ]:
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo
from concurrent.futures import ThreadPoolExecutor
from math import gcd
import json, hashlib, html, base64, platform, time, io
from functools import lru_cache
from math import ceil
from scipy.signal import welch, correlate
from scipy.spatial import cKDTree
import nbformat
import numpy as np
import pandas as pd
import soundfile as sf
import scipy
from scipy.signal import resample_poly
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown, Audio

DEVELOPMENT_DIR = Path('C:/Users/minhl/Downloads/Development_data/Development_data')
SPECIES_IDS = [1, 2, 6, 8]
DOMAINS = [1, 2, 3, 4]  # Domain của bộ chính, dùng cho cả ba split.
D5_ALBOPICTUS_CLIPS = 1374
SEED = 42
PROTOCOL_VERSION = 'A_v1'
# Nếu ZIP còn trên máy, đối chiếu checksum công bố để xác minh bản phát hành.
DEVELOPMENT_ARCHIVE = DEVELOPMENT_DIR.parent.parent/'Development_data.zip'
EXPECTED_ARCHIVE_MD5 = 'dbf35b9066e82cdee5461fca138f4986'
VALIDATION_GROUP_FRACTION = 0.20
MIN_VALIDATION_ELIGIBLE_CLIPS = 10
# Domain test chưa xuất hiện ở training của từng loài.
TEST_DOMAINS = {1:3, 2:2, 6:4, 8:2}
NEAR_MAX_ANCHORS = 128
NEAR_MIN_AC_RMS = 1e-5
NEAR_SPECTRAL_DISTANCE = 0.18
NEAR_SHORT_CORRELATION = 0.9999
NEAR_EXTENDED_CORRELATION = 0.999
NEAR_METHOD_VERSION = 'spectral_candidates_waveform_verification_v1' 
SAMPLE_RATE = 8000
WINDOW_MS = 300
HOP_MS = 150
WORKERS = 4

SPECIES_NAMES = {1:'Aedes aegypti', 2:'Aedes albopictus', 6:'Anopheles dirus', 8:'Anopheles minimus'}
# label_id 0..3 lưu rõ trong manifest, không suy ra từ thứ tự bảng.
LABEL_IDS = {species:i for i,species in enumerate(SPECIES_IDS)}
SPLIT_ORDER = ['train','validation','test']
DUPLICATE_PRIORITY = {'test':0, 'validation':1, 'train':2}
CLIPPING_LEVEL = 0.999

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p/'notebooks').is_dir() and (p/'data/manifests').is_dir())
AUDIT_DIR = ROOT/'results/A/biodcase'
OUT = ROOT/'results/A/preparation'
TABLES, FIGURES = OUT/'tables', OUT/'figures'
MANIFESTS = ROOT/'data/manifests'
for folder in [TABLES, FIGURES, MANIFESTS]: folder.mkdir(parents=True,exist_ok=True)
CLIP_MANIFEST = MANIFESTS/'biodcase_four_species_clips.csv'
WINDOW_MANIFEST = MANIFESTS/'biodcase_four_species_windows.csv'
assert SAMPLE_RATE>0 and 0<HOP_MS<=WINDOW_MS
WINDOW_SAMPLES = round(SAMPLE_RATE*WINDOW_MS/1000)
HOP_SAMPLES = round(SAMPLE_RATE*HOP_MS/1000)
assert WINDOW_SAMPLES>0 and HOP_SAMPLES>0
assert set(SPECIES_IDS)<=set(SPECIES_NAMES)
assert 5 not in DOMAINS, 'D5 chỉ được chọn qua biến bổ sung training.'
assert 2 in SPECIES_IDS and D5_ALBOPICTUS_CLIPS>=0
ORDER = [SPECIES_NAMES[s] for s in SPECIES_IDS]
sns.set_theme(style='whitegrid')
plt.rcParams.update({'figure.dpi':110,'axes.spines.top':False,'axes.spines.right':False})
pd.set_option('display.max_columns',20)

def table(frame,name):
    frame.to_csv(TABLES/f'{name}.csv',index=False,encoding='utf-8-sig')
    return frame

def plot_done(name):
    plt.tight_layout()
    plt.savefig(FIGURES/f'{name}.png',dpi=160,bbox_inches='tight')
    plt.show()
    plt.close()

print('Dữ liệu:', DEVELOPMENT_DIR)
print('Kết quả:', OUT)
print(f'Window: {WINDOW_SAMPLES} sample; hop: {HOP_SAMPLES} sample.')

## 2. Chọn D1–D4 và bổ sung Ae. albopictus từ training D5
Gán split bằng metadata chính thức. Giữ bộ D1–D4, chọn đúng số clip D5 đã khai báo, không hoàn lại, không dùng D5 validation/test. Xuất danh sách ID bổ sung để truy vết.

Các nhóm trùng được kiểm tra trên toàn bộ dữ liệu đã chọn. Nếu clip bổ sung không hợp lệ hoặc trùng, nó vẫn bị loại và số thực giữ được phải được báo cáo, không bỏ qua quy tắc làm sạch để đủ quota. Test chỉ đọc để audit, không dùng chọn frontend/model.

In [ ]:
audit_path=AUDIT_DIR/'audio_audit.csv'
assert audit_path.exists(), 'Chạy notebook 02_A_biodcase_analysis.ipynb trước.'
audit=pd.read_csv(audit_path,keep_default_na=False)
parsed=audit.clip_id.str.extract(r'^S_(?P<species_id>\d+)_D_(?P<domain_id>\d+)_(?P<clip_index>\d+)$')
assert not parsed.isna().any().any()
audit=pd.concat([audit,parsed.astype(int)],axis=1)
metadata=DEVELOPMENT_DIR/'metadata'
split_files={'train':'Training_ids.txt','validation':'Validation_ids.txt','test':'Test_ids.txt'}
split_ids={split:{Path(x.strip()).stem for x in (metadata/file).read_text().splitlines() if x.strip()}
           for split,file in split_files.items()}
for i,left in enumerate(SPLIT_ORDER):
    for right in SPLIT_ORDER[i+1:]: assert not split_ids[left]&split_ids[right]
assignment={clip:split for split,values in split_ids.items() for clip in values}
audit['original_split']=audit.clip_id.map(assignment)
audit['split']=audit.original_split
base_clips=audit.loc[audit.species_id.isin(SPECIES_IDS)&audit.domain_id.isin(DOMAINS)].copy()
base_clips['subset_role']='base_d1_d4'
d5_pool=audit.loc[audit.species_id.eq(2)&audit.domain_id.eq(5)&audit.split.eq('train')].copy()
d5_pool=d5_pool.sort_values(['species_id','domain_id','clip_index']).reset_index(drop=True)
assert D5_ALBOPICTUS_CLIPS<=len(d5_pool), 'Số clip yêu cầu vượt training pool D5.'
rng=np.random.default_rng(SEED)
selected=np.sort(rng.choice(len(d5_pool),size=D5_ALBOPICTUS_CLIPS,replace=False))
supplement=d5_pool.iloc[selected].copy()
supplement['subset_role']='supplement_d5_albopictus_train'
assert supplement.split.eq('train').all() and supplement.species_id.eq(2).all()
clips=pd.concat([base_clips,supplement],ignore_index=True)
clips['species']=clips.species_id.map(SPECIES_NAMES)
clips['label_id']=clips.species_id.map(LABEL_IDS)
clips['domain']='D'+clips.domain_id.astype(str)
clips['path']=[str(DEVELOPMENT_DIR/'raw_audio'/f'{clip}.wav') for clip in clips.clip_id]
assert clips.split.notna().all() and clips.clip_id.is_unique and not clips.empty
assert clips.error.eq('').all(), 'Inventory có lỗi audio: kiểm tra notebook 02.'
clips['original_seconds']=clips.seconds
base_counts=pd.crosstab(base_clips.species_id,base_clips.split).reindex(index=SPECIES_IDS,columns=SPLIT_ORDER,fill_value=0)
base_counts.index=ORDER
base_counts.index.name='species'
before_counts=pd.crosstab(clips.species,clips.split).reindex(index=ORDER,columns=SPLIT_ORDER,fill_value=0)
before_summary=clips.groupby(['species','split']).agg(clips=('clip_id','size'),minutes=('seconds',lambda s:s.sum()/60)).reset_index()
table(before_summary,'before_by_species_split')
table(base_counts.reset_index(),'base_d1_d4_before_counts')
table(supplement[['clip_id','species_id','domain_id','original_split','seconds','sha256']],'selected_d5_clips')
display(before_counts)
print(f'Bộ chính trước làm sạch: {len(base_clips):,} clip.')
print(f'Bổ sung {len(supplement):,}/{len(d5_pool):,} clip training D5 bằng seed {SEED}.')
print(f'Tổng trước xử lý: {len(clips):,} clip, {clips.seconds.sum()/60:.2f} phút.')


## 3. Kiểm tra tín hiệu và chuyển về mono 8 kHz
`prepare_audio()` được dùng thống nhất khi audit, minh họa và đọc window. Nếu nhiều kênh thì lấy trung bình; resample bằng `resample_poly` có lọc chống alias. Giữ biên độ số, chưa peak-normalize từng clip, thêm nhiễu hoặc pitch shift.

Kiểm tra tín hiệu rỗng/không hữu hạn/toàn zero, peak, RMS và tỷ lệ sample gần full scale. Chỉ audio không hợp lệ hoặc toàn zero bị loại tự động; âm lượng thấp/clipping được ghi nhận. RMS không phải SNR hoặc bằng chứng có muỗi.

Hash ba mức: byte file, mẫu giải mã gốc và mono 8 kHz float32. Mức cuối tìm tín hiệu trùng hoàn toàn sau tiền xử lý; chưa tìm mọi đoạn con, đổi gain hoặc gần trùng.

In [ ]:
def prepare_audio(path):
    native,sr=sf.read(str(path),dtype='float64',always_2d=True)
    if len(native)==0: raise ValueError('empty_audio')
    if not np.isfinite(native).all(): raise ValueError('nonfinite_audio')
    mono=native.mean(axis=1)
    if sr!=SAMPLE_RATE:
        factor=gcd(int(sr),SAMPLE_RATE)
        mono=resample_poly(mono,SAMPLE_RATE//factor,int(sr)//factor)
    canonical=np.asarray(mono,dtype='<f4')
    if not np.isfinite(canonical).all(): raise ValueError('nonfinite_after_resample')
    return native,int(sr),canonical

input_rows=clips[['clip_id','path','bytes','sha256']].to_dict('records')
def inspect_clip(row):
    result={'clip_id':row['clip_id'],'verified_sha256':'','pcm_sha256':'','canonical_sha256':'',
            'processed_frames':0,'processed_seconds':0.,'peak':np.nan,'rms_dbfs':np.nan,
            'clipped_sample_pct':np.nan,'all_zero':False,'decode_error':'','input_changed':False}
    try:
        st=Path(row['path']).stat()
        with open(row['path'],'rb') as handle: digest=hashlib.file_digest(handle,'sha256').hexdigest()
        result['verified_sha256']=digest
        result['input_changed']=bool(st.st_size!=int(row['bytes']) or digest!=row['sha256'])
        native,sr,signal=prepare_audio(row['path'])
        ph=hashlib.sha256(f'{sr}|{native.shape[1]}|{len(native)}|'.encode()+np.asarray(native,dtype='<f8').tobytes()).hexdigest()
        ch=hashlib.sha256(f'{SAMPLE_RATE}|1|{len(signal)}|'.encode()+signal.tobytes()).hexdigest()
        peak=float(np.max(np.abs(signal)))
        rms=float(np.sqrt(np.mean(signal.astype('float64')**2)))
        result.update(pcm_sha256=ph,canonical_sha256=ch,processed_frames=len(signal),
            processed_seconds=len(signal)/SAMPLE_RATE,peak=peak,rms_dbfs=20*np.log10(max(rms,1e-15)),
            clipped_sample_pct=100*float(np.mean(np.abs(signal)>=CLIPPING_LEVEL)),all_zero=bool(np.all(signal==0)))
    except Exception as exc: result['decode_error']=f'{type(exc).__name__}: {exc}'
    return result

print(f'Đọc và kiểm tra {len(input_rows):,} clip...',flush=True)
with ThreadPoolExecutor(max_workers=WORKERS) as pool:
    signal_audit=pd.DataFrame(list(pool.map(inspect_clip,input_rows)))
clips=clips.merge(signal_audit,on='clip_id',validate='one_to_one')
table(signal_audit,'signal_audit')
assert not clips.input_changed.any(), 'WAV khác inventory: chạy lại notebook 02.'
qc_summary=pd.DataFrame([
    {'check':'Lỗi giải mã / tín hiệu không hợp lệ','clips':int(clips.decode_error.ne('').sum()),'action':'Loại'},
    {'check':'Tín hiệu sau tiền xử lý toàn zero','clips':int(clips.all_zero.sum()),'action':'Loại'},
    {'check':f'Có sample |x| >= {CLIPPING_LEVEL}','clips':int(clips.clipped_sample_pct.gt(0).sum()),'action':'Gắn cờ, chưa loại'},
    {'check':'RMS dưới -60 dBFS','clips':int(clips.rms_dbfs.lt(-60).sum()),'action':'Mô tả, chưa loại'}])
table(qc_summary,'quality_summary')
display(qc_summary)
print('Đã kiểm tra checksum và đọc tín hiệu xong.')

## 4. Nhóm trùng và nhãn mâu thuẫn
Nối các clip vào cùng nhóm nếu trùng byte, mẫu gốc hoặc mono 8 kHz float32. Cách ly toàn bộ nhóm có nhiều nhãn loài. Đối chiếu thêm bảng mâu thuẫn trên **toàn bộ BioDCASE** để không bỏ sót nhãn thứ ba ngoài subset.

Nhóm cùng nhãn giữ một đại diện theo **test → validation → train**. Nếu cùng split, chọn domain ID và clip index nhỏ hơn để tái lập. Bước làm sạch này chưa chia lại tập: thứ tự ưu tiên áp dụng cho split tham chiếu. Mục 7 mới tạo split nghiên cứu A_v1. File bị loại vẫn ở manifest với lý do và ID đại diện; WAV không bị xóa.

In [ ]:
parent={clip:clip for clip in clips.clip_id}
def find(clip):
    while parent[clip]!=clip:
        parent[clip]=parent[parent[clip]]
        clip=parent[clip]
    return clip
def union(left,right):
    a,b=find(left),find(right)
    if a!=b: parent[max(a,b)]=min(a,b)
for column in ['verified_sha256','pcm_sha256','canonical_sha256']:
    for _,group in clips.loc[clips[column].ne('')].groupby(column):
        members=group.clip_id.tolist()
        for member in members[1:]: union(members[0],member)
clips['duplicate_group']=clips.clip_id.map(find)
conflict_path=AUDIT_DIR/'tables/conflicting_species_duplicates.csv'
assert conflict_path.exists(), 'Thiếu bảng nhãn mâu thuẫn từ notebook 02.'
global_conflicts=set(pd.read_csv(conflict_path).sha256)
local_conflicts=set(clips.groupby('duplicate_group').species_id.nunique().loc[lambda s:s>1].index)
conflict_groups=local_conflicts|set(clips.loc[clips.verified_sha256.isin(global_conflicts),'duplicate_group'])
clips['status']='keep'
clips['exclusion_reason']=''
clips['duplicate_of']=''
invalid=clips.decode_error.ne('')|clips.all_zero|clips.processed_frames.le(0)
clips.loc[invalid,['status','exclusion_reason']]=['exclude','invalid_audio']
conflict=clips.duplicate_group.isin(conflict_groups)
clips.loc[conflict,['status','exclusion_reason']]=['exclude','conflicting_species']
clips['split_priority']=clips.split.map(DUPLICATE_PRIORITY)
eligible=clips.loc[clips.status.eq('keep')].sort_values(['split_priority','domain_id','clip_index','clip_id'])
representatives=eligible.drop_duplicates('duplicate_group').set_index('duplicate_group').clip_id
is_duplicate=clips.status.eq('keep') & clips.clip_id.ne(clips.duplicate_group.map(representatives))
clips.loc[is_duplicate,'status']='exclude'
clips.loc[is_duplicate,'exclusion_reason']='duplicate_signal'
clips.loc[is_duplicate,'duplicate_of']=clips.loc[is_duplicate,'duplicate_group'].map(representatives)
clips['processed_sample_rate']=SAMPLE_RATE
clips['processed_channels']=1
clips['short_window']=clips.processed_frames.lt(WINDOW_SAMPLES)
clips['possible_windows']=np.where(clips.processed_frames>=WINDOW_SAMPLES,
    1+(clips.processed_frames-WINDOW_SAMPLES)//HOP_SAMPLES,0).astype('int64')
clips['full_windows']=np.where(clips.status.eq('keep'),clips.possible_windows,0).astype('int64')
clips['window_eligible']=clips.status.eq('keep') & ~clips.short_window
clips['window_exclusion_reason']=np.select(
    [clips.status.eq('exclude'),clips.short_window],['clip_excluded','shorter_than_window'],default='')
kept=clips.loc[clips.status.eq('keep')].copy()
removed=clips.loc[clips.status.eq('exclude')].copy()
removal_summary=table(removed.groupby(['species','split','exclusion_reason']).size().rename('clips').reset_index(),'excluded_by_reason')
table(removed[['clip_id','species','domain','split','exclusion_reason','duplicate_of','duplicate_group']],'excluded_clips')
display(removal_summary)
after_counts=pd.crosstab(kept.species,kept.split).reindex(index=ORDER,columns=SPLIT_ORDER,fill_value=0)
display(after_counts)
print(f'Giữ {len(kept):,}/{len(clips):,} clip; loại {len(removed):,} clip.')

clips['reference_split']=clips.split
reference_counts=after_counts.copy()
reference_window_summary=kept.groupby(['species','split']).agg(clips=('clip_id','size'),windows=('full_windows','sum')).reset_index()
table(reference_window_summary,'reference_split_summary')


## 5. Dấu vết nguồn qua HumBugDB
Liên kết các clip trùng mẫu chính xác với metadata `name` của HumBugDB. Giá trị này là **dấu vết nguồn**, chưa phải ID cá thể hoặc phiên thu độc lập. Không tự chuyển nhãn giới tính hay gán lại loài.

Cùng `name` ở nhiều split tham chiếu được báo thành cảnh báo nguồn và lưu lại. Mục 7 gộp các clip liên quan khi tạo split nghiên cứu; cảnh báo của bản tham chiếu vẫn được giữ để đối chiếu. Clip chưa có nguồn vẫn ghi chưa xác minh, không mặc định mỗi clip là một nguồn độc lập.

In [ ]:
match_file=AUDIT_DIR/'tables/humbug_pcm_matches.csv'
humbug_metadata=ROOT/'data/metadata/humbugdb_zenodo_0_0_2.csv'
assert match_file.exists() and humbug_metadata.exists(), 'Thiếu bảng đối chiếu HumBugDB.'
matches=pd.read_csv(match_file,dtype={'id':str})
hmeta=pd.read_csv(humbug_metadata,dtype={'id':str},keep_default_na=False)
links=matches[['clip_id','id']].drop_duplicates().merge(hmeta[['id','name']],on='id',how='left',validate='many_to_one')
links=links.loc[links.clip_id.isin(clips.clip_id)].copy()
links['name']=links.name.fillna('').str.strip()
name_sets=links.loc[links.name.ne('')].groupby('clip_id').name.agg(lambda s:' | '.join(sorted(set(s))))
id_sets=links.groupby('clip_id').id.agg(lambda s:' | '.join(sorted(set(s))))
clips['humbug_ids']=clips.clip_id.map(id_sets).fillna('')
clips['source_name_proxy']=clips.clip_id.map(name_sets).fillna('')
clips['source_verified']=False
kept=clips.loc[clips.status.eq('keep')].copy()
kept_links=links.loc[links.name.ne('')].merge(kept[['clip_id','species','domain','split']],on='clip_id')
source_groups=kept_links.groupby(['species','name']).agg(clips=('clip_id','nunique'),
    splits=('split',lambda s:' | '.join(sorted(set(s)))),split_count=('split','nunique'),
    domains=('domain',lambda s:' | '.join(sorted(set(s))))).reset_index()
source_warnings=source_groups.loc[source_groups.split_count.gt(1)]
table(source_groups,'known_source_proxies')
table(source_warnings,'source_proxy_cross_split_warnings')
source_coverage=kept.groupby(['species','split']).agg(clips=('clip_id','size'),
    matched_source_clips=('source_name_proxy',lambda s:s.ne('').sum())).reset_index()
source_coverage['matched_source_pct']=100*source_coverage.matched_source_clips/source_coverage.clips
table(source_coverage,'source_proxy_coverage')
display(source_coverage.round(2))
print(f'Nhóm (loài, name) có mặt ở nhiều split: {len(source_warnings):,}.')
print('Đây là cảnh báo nguồn; chưa chứng minh độc lập và chưa khóa protocol đánh giá.')

reference_source_warnings=source_warnings.copy()
table(reference_source_warnings,'reference_source_proxy_warnings')


## 6. Kiểm tra đoạn gần trùng và các bản cắt khác nhau
Lấy tối đa 128 anchor/clip (đoạn 300 ms, cách mép ít nhất 32 ms), bỏ anchor có AC RMS < 1e-5 để không coi đoạn gần im lặng chung là bằng chứng nguồn. So sánh đặc trưng phổ để lấy ứng viên, rồi tìm độ trễ và kiểm tra dạng sóng sau bỏ DC, cho phép khác gain/dấu.

Một liên kết mạnh cần tương quan đoạn 300 ms ≥ 0,9999 và kiểm tra mở rộng tối thiểu 600 ms, tối đa 1 s có tương quan ≥ 0,999. Cặp đó được cùng nhóm khi chia tập; chưa tự xóa clip chỉ vì gần trùng. Các cặp tương quan thấp hơn được ghi để xem xét, không coi chắc chắn là bản sao.

Đây là tìm kiếm có giới hạn anchor/ứng viên và độ trễ ±1 s quanh anchor bên kia, không phải bảo đảm đã tìm mọi dạng cắt, nhiễu hoặc resample. Cache gắn với hash nội dung và tham số tìm kiếm.

In [ ]:
near_clips=kept.sort_values('clip_id').reset_index(drop=True)
near_paths=near_clips.set_index('clip_id').path.to_dict()
sr=SAMPLE_RATE
length=round(.300*SAMPLE_RATE)
near_settings={'method':NEAR_METHOD_VERSION,'sr':sr,'length':length,'anchors':NEAR_MAX_ANCHORS,
    'min_ac_rms':NEAR_MIN_AC_RMS,'spectral_distance':NEAR_SPECTRAL_DISTANCE,
    'short_corr':NEAR_SHORT_CORRELATION,'extended_corr':NEAR_EXTENDED_CORRELATION}
near_signature=hashlib.sha256((near_clips[['clip_id','canonical_sha256']].to_csv(index=False)+repr(near_settings)).encode()).hexdigest()
near_file=TABLES/'near_duplicate_pairs.csv'
near_cache_file=TABLES/'near_search_summary.csv'
cache_valid=False
if near_file.exists() and near_cache_file.exists():
    previous=pd.read_csv(near_cache_file,keep_default_na=False).set_index('key').value.to_dict()
    cache_valid=(previous.get('input_signature')==near_signature and previous.get('pairs_sha256')==hashlib.sha256(near_file.read_bytes()).hexdigest())

@lru_cache(maxsize=48)
def near_signal(clip):
    return prepare_audio(near_paths[clip])[2]

if cache_valid:
    pairs=pd.read_csv(near_file)
    anchor_count=int(previous['anchors'])
    candidate_count=int(previous['candidate_pairs'])
    print('Dùng cache gần trùng có kiểm tra hash:',len(pairs),'cặp cần xem.',flush=True)
else:
    features=[]; anchors=[]
    started=time.monotonic()
    for i,row in enumerate(near_clips.itertuples()):
        x=near_signal(row.clip_id)
        if len(x)<length: continue
        margin=round(.032*SAMPLE_RATE)
        if len(x)<length+2*margin: continue
        count=min(NEAR_MAX_ANCHORS,1+(len(x)-length-2*margin)//length)
        starts=np.unique(np.linspace(margin,len(x)-length-margin,max(1,count),dtype=int))
        for start in starts:
            z=x[start:start+length].astype('float64'); z-=z.mean()
            if np.sqrt(np.mean(z*z))<NEAR_MIN_AC_RMS: continue
            _,power=welch(z,fs=sr,nperseg=256,noverlap=128)
            f=np.log10(np.maximum(power[2:113],1e-20))
            f=np.interp(np.linspace(0,len(f)-1,40),np.arange(len(f)),f)
            f-=f.mean(); f/=max(np.linalg.norm(f),1e-12)
            features.append(f); anchors.append((row.clip_id,int(start)))
        if i%400==0: print('Fingerprint',i,'/',len(near_clips),'seconds',round(time.monotonic()-started),flush=True)
    features=np.asarray(features)
    tree=cKDTree(features)
    dist,idx=tree.query(features,k=12,workers=4)
    candidates={}
    for i,(clip,start) in enumerate(anchors):
        seen=0
        for d,j in zip(dist[i,1:],idx[i,1:]):
            other,other_start=anchors[j]
            if other==clip or d>NEAR_SPECTRAL_DISTANCE: continue
            key=tuple(sorted((clip,other)))
            item=(float(d),clip,start,other,other_start)
            if key not in candidates or d<candidates[key][0]: candidates[key]=item
            seen+=1
            if seen>=2: break
    print('Anchors',len(anchors),'candidate clip pairs',len(candidates),flush=True)
    
    def verify(clip,start,other,other_start):
        query=near_signal(clip)[start:start+length].astype('float64')
        target=near_signal(other)
        lo=max(0,other_start-SAMPLE_RATE); hi=min(len(target),other_start+length+SAMPLE_RATE)
        segment=target[lo:hi].astype('float64')
        if len(segment)<length: return 0.,0,0.
        query-=query.mean(); qnorm=np.sqrt(np.sum(query*query))
        if qnorm<1e-10: return 0.,0,0.
        sums=np.r_[0,np.cumsum(segment)]; squares=np.r_[0,np.cumsum(segment*segment)]
        sx=sums[length:]-sums[:-length]; sx2=squares[length:]-squares[:-length]
        denom=qnorm*np.sqrt(np.maximum(sx2-sx*sx/length,1e-20))
        values=correlate(segment,query,mode='valid',method='fft')/denom
        best=int(np.argmax(np.abs(values)))
        actual=segment[best:best+length]; actual=actual-actual.mean()
        corr=float(abs(np.dot(actual,query)/(np.linalg.norm(actual)*qnorm))) if np.linalg.norm(actual)>1e-10 else 0.
        corr=min(1.,corr)
        return corr,lo+best,float(np.sqrt(max(0.,1-corr*corr)))
    
    accepted=[]; near_review=[]
    for i,item in enumerate(sorted(candidates.values())):
        d,clip,start,other,other_start=item
        corr,match_start,relative_residual=verify(clip,start,other,other_start)
        if corr>=.995:
            row={'clip_a':clip,'start_a':start,'clip_b':other,'start_b':match_start,
                 'segment_samples':length,'spectral_distance':d,'abs_correlation':corr,
                 'relative_residual':relative_residual,'strong_link':corr>=NEAR_SHORT_CORRELATION}
            near_review.append(row)
            if row['strong_link']: accepted.append(row)
        if i%1000==0: print('Verify',i,'/',len(candidates),'strong',len(accepted),'seconds',round(time.monotonic()-started),flush=True)

    pairs=pd.DataFrame(near_review,columns=['clip_a','start_a','clip_b','start_b','segment_samples','spectral_distance','abs_correlation','relative_residual','strong_link'])
    extended=[]; lengths=[]
    for row in pairs.itertuples():
        a=near_signal(row.clip_a); b=near_signal(row.clip_b); offset=row.start_b-row.start_a
        left=max(0,-offset,row.start_a-round(.35*SAMPLE_RATE)); right=min(len(a),len(b)-offset,left+SAMPLE_RATE)
        qa=a[left:right].astype('float64'); qb=b[left+offset:right+offset].astype('float64')
        qa-=qa.mean();qb-=qb.mean();denom=np.linalg.norm(qa)*np.linalg.norm(qb)
        extended.append(abs(np.dot(qa,qb)/denom) if denom>1e-10 else 0.)
        lengths.append(len(qa))
    pairs['extended_abs_correlation']=extended
    pairs['extended_samples']=lengths
    pairs['strong_link']=pairs.abs_correlation.ge(NEAR_SHORT_CORRELATION)&pairs.extended_abs_correlation.ge(NEAR_EXTENDED_CORRELATION)&pairs.extended_samples.ge(round(.6*SAMPLE_RATE))
    pairs.to_csv(near_file,index=False)
    anchor_count=len(anchors); candidate_count=len(candidates)
    cache_rows={'input_signature':near_signature,'pairs_sha256':hashlib.sha256(near_file.read_bytes()).hexdigest(),
        'anchors':anchor_count,'candidate_pairs':candidate_count,'review_pairs':len(pairs),'strong_pairs':int(pairs.strong_link.sum()),
        'method':NEAR_METHOD_VERSION,'search_complete_for_all_possible_offsets':'False'}
    table(pd.DataFrame(cache_rows.items(),columns=['key','value']),'near_search_summary')
strong_pairs=pairs.loc[pairs.strong_link].copy()
table(strong_pairs,'strong_waveform_links')
display(strong_pairs[['clip_a','clip_b','abs_correlation','extended_abs_correlation','extended_samples']].round(6))
print(f'{anchor_count:,} anchor, {candidate_count:,} cặp ứng viên; {len(strong_pairs)} liên kết mạnh.')
near_signal.cache_clear()

## 7. Chốt cách chia nghiên cứu A_v1 theo nhóm và domain
Giữ `reference_split` của bản tham chiếu. Cột `split` mới dùng để chạy baseline: giữ domain test theo loài (**Ae. aegypti D3; Ae. albopictus D2; An. dirus D4; An. minimus D2**). Nếu một nhóm liên quan chạm test, toàn nhóm vào test.

Các clip có cùng `name` đối chiếu hoặc liên kết dạng sóng mạnh luôn ở cùng split. Chọn validation từ nhóm còn lại ngoài D5 bằng seed cố định, mục tiêu 20% nhóm và tối thiểu 10 clip đủ window nếu dữ liệu cho phép. Giữ D5 chỉ training. Với Ae. albopictus, D1 chỉ có một nguồn đối chiếu nên toàn bộ nguồn vào validation; train là D5, test là D2. Số clip validation tăng nhưng vẫn chỉ một nguồn D1, chưa giải quyết thiếu đa dạng nguồn.

Nhóm `unverified_clip` là đơn vị tạm khi thiếu thông tin, không chứng minh độc lập bản ghi/cá thể. Bản A_v1 bảo đảm không giao nhau **ở nhóm đã biết/phát hiện**; report công bố giới hạn còn lại. Việc chia lại diễn ra trước khi huấn luyện model, không dùng kết quả test để lựa chọn.

**Kết luận rà thiết kế:** giữ A_v1 như thí nghiệm chuyển domain theo loài. Trong pool đang chọn, nếu giữ D2 làm test và D5 chỉ training cho Ae. albopictus thì nguồn D1 còn lại chỉ là một nhóm đã biết; không có đủ nguồn đã xác minh để tạo thêm validation đa nguồn theo các ràng buộc này. Đổi tỷ lệ clip hoặc seed không tạo thêm nguồn độc lập. Không kích hoạt protocol phân loại trong cùng domain cho cả bốn loài khi chưa có bằng chứng đủ nguồn.

`source_domain_coverage.csv` và `split_design_review.csv` ghi số liệu/giới hạn theo từng loài; `protocol_options.csv` ghi phương án được chọn và phương án chưa đủ cơ sở. Nhóm chưa xác minh không được tính là cá thể độc lập. Rà soát này chỉ kết luận trên pool hiện tại, không khẳng định toàn bộ dữ liệu bên ngoài không có phương án khác.

In [ ]:
kept=clips.loc[clips.status.eq('keep')].copy()
leak_parent={clip:clip for clip in kept.clip_id}
def leak_find(clip):
    while leak_parent[clip]!=clip:
        leak_parent[clip]=leak_parent[leak_parent[clip]]
        clip=leak_parent[clip]
    return clip
def leak_union(left,right):
    a,b=leak_find(left),leak_find(right)
    if a!=b: leak_parent[max(a,b)]=min(a,b)
source_representatives={}
for row in kept.itertuples():
    for name in [x.strip() for x in row.source_name_proxy.split(' | ') if x.strip()]:
        if name in source_representatives: leak_union(row.clip_id,source_representatives[name])
        else: source_representatives[name]=row.clip_id
for row in strong_pairs.itertuples(): leak_union(row.clip_a,row.clip_b)
kept['leakage_group']=kept.clip_id.map(leak_find)
waveform_groups=set(kept.set_index('clip_id').loc[sorted(set(strong_pairs.clip_a)|set(strong_pairs.clip_b)),'leakage_group']) if len(strong_pairs) else set()
name_groups=set(kept.loc[kept.source_name_proxy.ne(''),'leakage_group'])
def group_basis(group):
    evidence=[]
    if group in name_groups:evidence.append('matched_name_proxy')
    if group in waveform_groups:evidence.append('strong_waveform_link')
    return ' + '.join(evidence) if evidence else 'unverified_clip'
kept['group_basis']=kept.leakage_group.map(group_basis)
test_groups=set(kept.loc[kept.domain_id.eq(kept.species_id.map(TEST_DOMAINS)),'leakage_group'])
remaining=kept.loc[~kept.leakage_group.isin(test_groups)]
val_rng=np.random.default_rng(SEED)
val_groups=set()
for species_id,sub in remaining.groupby('species_id',sort=True):
    eligible=sub.loc[sub.domain_id.ne(5)].groupby('leakage_group').agg(
        clips=('clip_id','size'),eligible=('window_eligible','sum'),windows=('full_windows','sum'))
    eligible=eligible.loc[eligible.eligible.gt(0)]
    choices=sorted(eligible.index)
    choices=[choices[i] for i in val_rng.permutation(len(choices))]
    target=max(1,ceil(VALIDATION_GROUP_FRACTION*len(choices)))
    chosen=[]
    for group in choices:
        chosen.append(group)
        if len(chosen)>=target and eligible.loc[chosen].eligible.sum()>=MIN_VALIDATION_ELIGIBLE_CLIPS:break
    if not sub.domain_id.eq(5).any() and len(chosen)>=sub.leakage_group.nunique():chosen=chosen[:-1]
    val_groups.update(chosen)
kept['split']=np.where(kept.leakage_group.isin(test_groups),'test',np.where(kept.leakage_group.isin(val_groups),'validation','train'))
clips['leakage_group']=''
clips['group_basis']='excluded'
for column in ['split','leakage_group','group_basis']:
    mapping=kept.set_index('clip_id')[column]
    use=clips.status.eq('keep')
    clips.loc[use,column]=clips.loc[use,'clip_id'].map(mapping)
clips['split_protocol']=PROTOCOL_VERSION
kept=clips.loc[clips.status.eq('keep')].copy()
after_counts=pd.crosstab(kept.species,kept.split).reindex(index=ORDER,columns=SPLIT_ORDER,fill_value=0)
research_groups=kept.groupby(['species','split']).agg(clips=('clip_id','size'),
    groups=('leakage_group','nunique'),eligible_clips=('window_eligible','sum'),
    known_name_clips=('source_name_proxy',lambda s:s.ne('').sum()),windows=('full_windows','sum'),
    minutes=('processed_seconds',lambda s:s.sum()/60)).reset_index()
table(research_groups,'research_split_summary')
table(kept.groupby(['species','group_basis']).size().rename('clips').reset_index(),'source_evidence_coverage')
table(kept[['clip_id','reference_split','split','leakage_group','group_basis']],'split_assignments')
research_links=links.loc[links.name.ne('')].merge(kept[['clip_id','species','domain','split']],on='clip_id')
source_groups=research_links.groupby(['species','name']).agg(clips=('clip_id','nunique'),
    splits=('split',lambda s:' | '.join(sorted(set(s)))),split_count=('split','nunique'),
    domains=('domain',lambda s:' | '.join(sorted(set(s))))).reset_index()
source_warnings=source_groups.loc[source_groups.split_count.gt(1)]
table(source_groups,'known_source_proxies')
table(source_warnings,'source_proxy_cross_split_warnings')
source_coverage=kept.groupby(['species','split']).agg(clips=('clip_id','size'),
    matched_source_clips=('source_name_proxy',lambda s:s.ne('').sum())).reset_index()
source_coverage['matched_source_pct']=100*source_coverage.matched_source_clips/source_coverage.clips
table(source_coverage,'source_proxy_coverage')
provenance_evidence=pd.DataFrame([
    {'source':'BioDCASE release + task page','available':'species, domain, clip index, official split','missing':'individual ID, original recording/session mapping','url':'https://zenodo.org/records/20478577'},
    {'source':'Baseline paper v1','available':'domain definition and split protocol','missing':'per-clip mapping to original recording in material inspected','url':'https://arxiv.org/html/2603.20118v1'},
    {'source':'Exact PCM match to local HumBugDB','available':'Name matches counted below','missing':'individual independence not certified by name','url':'local metadata + match table'},
    {'source':'Waveform search','available':f'{len(strong_pairs)} strong links, matched intervals and correlations','missing':'exhaustive near-duplicate coverage and biological individual ID','url':'tables/strong_waveform_links.csv'}])
provenance_evidence.loc[2,'available']=f'{int(kept.source_name_proxy.ne("").sum())} kept clips have a name proxy'
archive_md5='not_available'
if DEVELOPMENT_ARCHIVE.exists():
    with DEVELOPMENT_ARCHIVE.open('rb') as handle:
        archive_md5=hashlib.file_digest(handle,'md5').hexdigest()
    assert archive_md5==EXPECTED_ARCHIVE_MD5, 'ZIP khác checksum Zenodo v2; kiểm tra phiên bản trước khi tiếp tục.'
    archive_status='MD5 matches published Zenodo v2'
else:
    archive_status='ZIP not present; no archive-level claim for this run'
provenance_evidence=pd.concat([provenance_evidence,pd.DataFrame([{
    'source':'Local Development_data.zip integrity','available':archive_status+'; '+archive_md5,
    'missing':'Archive checksum does not certify biological label or individual independence',
    'url':'https://zenodo.org/records/20478577'}])],ignore_index=True)
table(provenance_evidence,'provenance_evidence')
modality_rows=[]
for (species,domain),sub in kept.groupby(['species','domain']):
    named=int(sub.source_name_proxy.ne('').sum())
    modality_rows.append({'species':species,'domain':domain,'clips':len(sub),'humbug_matched_clips':named,
        'sensor_evidence':'matched acoustic HumBug records for matched subset' if named else 'sensor modality not verified from provided mapping',
        'unmapped_clips':len(sub)-named,'microphone_only_certified_for_whole_domain':False})
modality_evidence=table(pd.DataFrame(modality_rows),'sensor_modality_evidence')


# Rà khả năng chia theo nguồn/domain trên đúng pool đã chọn, không chia lại bằng seed để tăng số nguồn.
source_domain_coverage = table(kept.groupby(['species','domain','split']).agg(
    clips=('clip_id','size'), eligible_clips=('window_eligible','sum'),
    evidence_groups=('leakage_group','nunique'),
    known_name_clips=('source_name_proxy',lambda s:s.ne('').sum()),
    unverified_source_clips=('group_basis',lambda s:s.eq('unverified_clip').sum()),
    minutes=('processed_seconds',lambda s:s.sum()/60), windows=('full_windows','sum')
).reset_index(),'source_domain_coverage')
review_notes = {
    1: ('Giữ A_v1 để thử chuyển D1/D4 sang D3',
        'Validation cùng domain với train; nguồn D3 test chưa xác minh độc lập đầy đủ.'),
    2: ('Giữ có điều kiện; validation là hạn chế chính',
        'Train chỉ D5, validation chỉ một nguồn D1, test D2; nhãn và domain gắn mạnh; cảm biến D5 chưa xác minh.'),
    6: ('Giữ A_v1; cần theo dõi chuyển D1 sang D4',
        'Train ít window nhất và chỉ D1; validation D1 không đại diện mọi khó khăn trên D4.'),
    8: ('Giữ A_v1; theo dõi tỷ trọng D4 trong training',
        'Nhiều window từ một số clip D4 dài; nguồn test D2 và cá thể chưa được xác minh đầy đủ.')}
review_rows = []
for species_id in SPECIES_IDS:
    sub = kept.loc[kept.species_id.eq(species_id)]
    tr = sub.loc[sub.split.eq('train')]
    va = sub.loc[sub.split.eq('validation')]
    te = sub.loc[sub.split.eq('test')]
    domain_windows = tr.groupby('domain').full_windows.sum()
    decision, limitation = review_notes[species_id]
    review_rows.append(dict(species=SPECIES_NAMES[species_id],
        train_domains=' | '.join(sorted(tr.domain.unique())),
        validation_domains=' | '.join(sorted(va.domain.unique())),
        test_domains=' | '.join(sorted(te.domain.unique())),
        train_eligible_clips=int(tr.window_eligible.sum()),
        validation_eligible_clips=int(va.window_eligible.sum()),
        test_eligible_clips=int(te.window_eligible.sum()),
        validation_evidence_groups=int(va.loc[va.window_eligible,'leakage_group'].nunique()),
        unverified_source_clips=int(sub.group_basis.eq('unverified_clip').sum()),
        largest_train_domain_window_share_pct=100*domain_windows.max()/domain_windows.sum(),
        decision=decision, limitation=limitation))
split_design_review = table(pd.DataFrame(review_rows),'split_design_review')
protocol_options = table(pd.DataFrame([
    dict(option='A_v1: chuyển domain theo từng loài',status='Kích hoạt cho baseline thăm dò',
         reason='Giữ nguyên nhóm đã biết/phát hiện, không dùng điểm model; công bố thiếu metadata và validation albopictus một nguồn.'),
    dict(option='Phân loại trong cùng domain, giữ riêng nguồn cho đủ bốn loài',status='Chưa kích hoạt',
         reason='Trong pool và ràng buộc hiện tại, chưa đủ nguồn đã xác minh cho cả bốn loài. Không chia lại window/clip cùng nguồn để tạo validation giả độc lập.')
]),'protocol_options')
display(split_design_review[['species','train_domains','validation_domains','test_domains',
    'validation_evidence_groups','decision','limitation']])
display(protocol_options)

display(research_groups.round(3))
print('Nhóm name đi qua split nghiên cứu:',len(source_warnings))
print('D5 chỉ training; source grouping chỉ bảo đảm theo bằng chứng đang có, không phải ID cá thể.')

## 8. Rà chất lượng tín hiệu và quyết định giữ/loại
Kiểm tra tất cả clip giữ lại bị gắn cờ, phân biệt chạm biên trong WAV gốc với overshoot sau resample. Đo tỷ lệ và chuỗi mẫu gần biên dài nhất. Audio hữu hạn, không toàn zero được giữ khi clipping chỉ thưa hoặc âm lượng thấp; không có bằng chứng để loại cả bản ghi chỉ bằng RMS.

Biểu đồ và audio minh họa chỉ từ training của bản nghiên cứu. Có đoạn phát thử trong report để kiểm tra lại; kiểm tra này không tái xác nhận loài/giới tính bằng tai và không đo SNR đã hiệu chuẩn. Quyết định kỹ thuật, lý do và cờ chất lượng được lưu cho từng clip.

In [ ]:
qc_candidates=kept.loc[kept.clipped_sample_pct.gt(0)|kept.rms_dbfs.lt(-60)].copy()
qc_rows=[]
for row in qc_candidates.itertuples():
    native,native_sr,processed=prepare_audio(row.path)
    native_mono=native.mean(axis=1)
    rail=np.abs(native_mono)>=CLIPPING_LEVEL
    edges=np.diff(np.r_[False,rail,False].astype(int))
    starts=np.flatnonzero(edges==1); ends=np.flatnonzero(edges==-1)
    qc_rows.append({'clip_id':row.clip_id,'species':row.species,'domain':row.domain,
        'reference_split':row.reference_split,'split':row.split,'native_peak':float(np.max(np.abs(native_mono))),
        'native_rail_pct':100*float(rail.mean()),'max_rail_run_samples':int(max(ends-starts,default=0)),
        'native_sample_rate':native_sr,'processed_peak':row.peak,'rms_dbfs':row.rms_dbfs,
        'decision':'retain_flagged','reason':'Giữ: tín hiệu hữu hạn, không toàn zero; âm lượng thấp/chạm biên thưa chưa đủ để loại cả clip',
        'species_content_reannotated':False})
qc_review=pd.DataFrame(qc_rows)
table(qc_review,'quality_review_decisions')
clips['qc_decision']='retain_unflagged'
clips.loc[clips.status.eq('exclude'),'qc_decision']='excluded_by_existing_rules'
clips.loc[clips.clip_id.isin(qc_candidates.clip_id),'qc_decision']='retain_flagged'
kept=clips.loc[clips.status.eq('keep')].copy()
train_review=qc_review.loc[qc_review.split.eq('train')]
example_ids=[]
if not train_review.empty:
    example_ids.extend([train_review.sort_values('rms_dbfs').iloc[0].clip_id,train_review.sort_values('native_rail_pct',ascending=False).iloc[0].clip_id])
for species_id in [1,2]:
    controls=kept.loc[kept.split.eq('train')&kept.species_id.eq(species_id)&kept.window_eligible]
    if not controls.empty:example_ids.append(controls.sort_values('clip_id').iloc[0].clip_id)
example_ids=list(dict.fromkeys(example_ids))
qc_audio_html=[]
fig,axes=plt.subplots(len(example_ids),2,figsize=(13,max(3,2.5*len(example_ids))),squeeze=False)
qc_examples=[]
for i,clip in enumerate(example_ids):
    row=kept.loc[kept.clip_id.eq(clip)].iloc[0]
    _,_,x=prepare_audio(row.path)
    peak=int(np.argmax(np.abs(x)))
    begin=max(0,min(peak-SAMPLE_RATE//2,len(x)-2*SAMPLE_RATE))
    end=min(len(x),begin+2*SAMPLE_RATE)
    snippet=x[begin:end]
    axes[i,0].plot(np.arange(len(snippet))/SAMPLE_RATE,snippet,lw=.6)
    axes[i,0].set_title(f'{row.species} | {clip} | {row.qc_decision}',fontsize=9)
    axes[i,0].set_xlabel('Giây trong đoạn minh họa');axes[i,0].set_ylabel('Biên độ số')
    freq,power=welch(snippet,fs=SAMPLE_RATE,nperseg=min(512,len(snippet)))
    axes[i,1].plot(freq,10*np.log10(np.maximum(power,1e-20)))
    axes[i,1].set_xlabel('Hz');axes[i,1].set_ylabel('PSD dB/Hz');axes[i,1].set_title('Phổ mô tả, không tự gán F0 hoặc giới tính',fontsize=9)
    buffer=io.BytesIO();sf.write(buffer,np.clip(snippet,-1,1),SAMPLE_RATE,format='WAV',subtype='PCM_16')
    payload=base64.b64encode(buffer.getvalue()).decode()
    qc_audio_html.append(f'<p>{html.escape(clip)} — chỉ playback giới hạn [-1,1], không tự tăng gain.</p><audio controls preload="none" src="data:audio/wav;base64,{payload}"></audio>')
    display(Audio(np.clip(snippet,-1,1),rate=SAMPLE_RATE,normalize=False))
    qc_examples.append({'clip_id':clip,'split':row.split,'start_sample':begin,'end_sample':end,'sample_rate':SAMPLE_RATE,'qc_decision':row.qc_decision})
plot_done('06_quality_review')
table(pd.DataFrame(qc_examples),'quality_review_examples')
display(qc_review.round(6))
print('Không loại thêm clip chỉ vì âm lượng thấp/clipping thưa; không tái gán nhãn sinh học bằng kiểm tra tín hiệu.')

## 9. Manifest clip và chỉ mục window
Manifest clip giữ cả file được chọn và bị loại. Manifest window chỉ chứa window từ clip giữ lại, đủ dài. `start_sample` tính từ đầu **tín hiệu đã chuyển về 8 kHz**, `end_sample` là cận phải không bao gồm sample đó.

Cắt riêng từng clip. Đuôi ngắn hơn window không được pad hoặc nối; phần chưa phủ được thống kê. Clip ngắn vẫn ở manifest để khảo sát window khác. Chưa nhân bản để cân bằng, tạo frontend hay augmentation. Thống kê chuẩn hóa học được sau này chỉ được fit trên training.

`full_windows` là số window thực được xuất của mỗi clip; không cắt bớt lớp lớn. `subset_role` phân biệt D1–D4 và dữ liệu bổ sung. Sampling/loss xử lý imbalance sẽ được thử sau trên cùng pool dữ liệu.

`reference_split` giữ bản tham chiếu; `split`, `leakage_group`, `group_basis` và `split_protocol` ghi rõ protocol A_v1. `source_verified=False` không được hiểu thành bằng chứng không rò rỉ.

In [ ]:
rows=[]
for row in kept.sort_values(['split_priority','species_id','domain_id','clip_index']).itertuples():
    for index in range(int(row.full_windows)):
        start=index*HOP_SAMPLES
        rows.append({'window_id':f'{row.clip_id}_w{index:06d}','clip_id':row.clip_id,
            'species':row.species,'species_id':row.species_id,'label_id':row.label_id,
            'domain':row.domain,'split':row.split,'reference_split':row.reference_split,'leakage_group':row.leakage_group,'group_basis':row.group_basis,'split_protocol':PROTOCOL_VERSION,'subset_role':row.subset_role,'path':row.path,'sample_rate':SAMPLE_RATE,
            'start_sample':start,'end_sample':start+WINDOW_SAMPLES,
            'start_seconds':start/SAMPLE_RATE,'end_seconds':(start+WINDOW_SAMPLES)/SAMPLE_RATE,
            'duplicate_group':row.duplicate_group})
windows=pd.DataFrame(rows,columns=['window_id','clip_id','species','species_id','label_id','domain','split','reference_split','leakage_group','group_basis','split_protocol','subset_role','path',
    'sample_rate','start_sample','end_sample','start_seconds','end_seconds','duplicate_group'])
clips['covered_samples']=np.where(clips.full_windows.gt(0),(clips.full_windows-1)*HOP_SAMPLES+WINDOW_SAMPLES,0)
clips['uncovered_tail_samples']=clips.processed_frames-clips.covered_samples
clips.loc[clips.status.eq('exclude'),['covered_samples','uncovered_tail_samples']]=0
keep_columns=['clip_id','clip_index','path','species','species_id','label_id','domain','domain_id','original_split','reference_split','split','leakage_group','group_basis','split_protocol','qc_decision',
    'subset_role','status','exclusion_reason','duplicate_of','duplicate_group','sha256','pcm_sha256','canonical_sha256',
    'sample_rate','channels','frames','original_seconds','processed_sample_rate','processed_channels','processed_frames',
    'processed_seconds','peak','rms_dbfs','clipped_sample_pct','all_zero','decode_error','short_window',
    'window_eligible','window_exclusion_reason','possible_windows','full_windows','covered_samples','uncovered_tail_samples',
    'humbug_ids','source_name_proxy','source_verified']
clips[keep_columns].sort_values(['species_id','domain_id','clip_index']).to_csv(CLIP_MANIFEST,index=False,encoding='utf-8-sig')
windows.to_csv(WINDOW_MANIFEST,index=False,encoding='utf-8-sig')
kept=clips.loc[clips.status.eq('keep')].copy()
window_summary=kept.groupby(['species','split']).agg(clips=('clip_id','size'),
    minutes=('processed_seconds',lambda s:s.sum()/60),short_clips=('short_window','sum'),
    window_eligible_clips=('window_eligible','sum'),windows=('full_windows','sum'),
    uncovered_seconds=('uncovered_tail_samples',lambda s:s.sum()/SAMPLE_RATE)).reset_index()
table(window_summary,'windows_by_species_split')
table(kept.loc[kept.short_window,['clip_id','species','split','processed_seconds','window_exclusion_reason']],'short_clips')
table(kept.groupby(['species','domain','split']).agg(clips=('clip_id','size'),
    minutes=('processed_seconds',lambda s:s.sum()/60),windows=('full_windows','sum')).reset_index(),'after_species_domain_split')
display(window_summary.round(3))
print(f'Đã xuất {len(windows):,} window; chỉ lưu chỉ mục, không ghi WAV nhỏ.')

## 10. Phân bố sau bổ sung và làm sạch
Biểu đồ đầu so sánh split tham chiếu sau làm sạch với split nghiên cứu A_v1; không biểu diễn số file bị loại. Số bị loại được báo riêng trong bảng làm sạch. Bảng bổ sung riêng giúp phân biệt tăng dữ liệu với loại bản trùng. Giữ toàn bộ window training hợp lệ, chấp nhận imbalance để so sánh phương pháp xử lý ở B–C.

Đánh giá riêng theo loài và domain vì D5 chỉ có trong training Ae. albopictus. Số window không thay thế cho số cá thể hay nguồn độc lập.

In [ ]:
fig,axes=plt.subplots(1,3,figsize=(16,5),sharey=True)
for axis,split in zip(axes,SPLIT_ORDER):
    comp=pd.DataFrame({'Tham chiếu':reference_counts[split],'Nghiên cứu':after_counts[split]})
    comp.plot.barh(ax=axis,color=['#B9C9CE','#247B91'])
    axis.set_title(split); axis.set_xlabel('Số clip'); axis.set_ylabel('')
    for container in axis.containers: axis.bar_label(container,padding=3,fontsize=8)
    axis.set_xlim(0,comp.to_numpy().max()*1.25)
fig.suptitle('Bản chia tham chiếu và bản nghiên cứu A_v1',y=1.02)
plot_done('01_before_after_clips')
training=window_summary.loc[window_summary.split.eq('train')].set_index('species').reindex(ORDER)
fig,axes=plt.subplots(1,2,figsize=(13,4))
training.minutes.plot.barh(ax=axes[0],color='#247B91')
axes[0].set_xlabel('Phút audio'); axes[0].set_ylabel(''); axes[0].set_title('Training: thời lượng sau làm sạch')
training.windows.plot.barh(ax=axes[1],color='#D29249')
axes[1].set_xlabel(f'Số window {WINDOW_MS} ms, hop {HOP_MS} ms'); axes[1].set_ylabel(''); axes[1].set_title('Training: số window dự kiến')
for axis in axes:
    axis.bar_label(axis.containers[0],fmt='%.1f',padding=3,fontsize=8)
    lo,hi=axis.get_xlim(); axis.set_xlim(lo,hi*1.15)
plot_done('02_training_duration_windows')
train_kept=kept.loc[kept.split.eq('train')]
domain_counts=pd.crosstab(train_kept.species,train_kept.domain).reindex(index=ORDER,columns=[f'D{i}' for i in sorted(set(DOMAINS)|set(train_kept.domain_id))],fill_value=0)
plt.figure(figsize=(8,4))
sns.heatmap(domain_counts,annot=True,fmt='d',cmap='Blues',cbar_kws={'label':'Số clip'})
plt.title('Training sau làm sạch: loài × domain'); plt.xlabel('Domain'); plt.ylabel('')
plot_done('03_training_domain_coverage')
plt.figure(figsize=(9,4))
sns.boxplot(data=train_kept,x='rms_dbfs',y='species',order=ORDER,showfliers=False,color='#8CC4CD')
plt.xlabel('RMS (dBFS, biên độ số)'); plt.ylabel(''); plt.title('Training: mức tín hiệu — không phải SNR')
plot_done('04_training_signal_levels')
train_ratios={'clips':float(training.clips.max()/training.clips.min()),
              'minutes':float(training.minutes.max()/training.minutes.min()),
              'windows':float(training.windows.max()/training.windows.min()) if training.windows.min()>0 else None}
display(Markdown(f'**Training:** chênh lệch lớn nhất/nhỏ nhất theo clip **{train_ratios["clips"]:.2f} lần**, theo thời lượng **{train_ratios["minutes"]:.2f} lần**.'))

supplement_outcome=clips.loc[clips.subset_role.ne('base_d1_d4')].groupby(['species','status','exclusion_reason']).agg(
    clips=('clip_id','size'),minutes=('processed_seconds',lambda s:s.sum()/60),windows=('full_windows','sum')).reset_index()
table(supplement_outcome,'d5_supplement_outcome')
display(supplement_outcome.round(3))
imbalance_table=training[['clips','minutes','window_eligible_clips','windows']].reset_index()
table(imbalance_table,'training_imbalance')
display(imbalance_table.round(3))


## 11. Đọc thử window bằng đúng quy trình
Chọn một window training mỗi loài để kiểm tra chỉ mục, độ dài và dải tín hiệu. Sóng và STFT được vẽ trên đúng số sample dự kiến đưa vào model. STFT chỉ minh họa, chưa chốt frontend.

Spectrogram dùng chung thang màu, không phải dB SPL hiệu chuẩn. Chỉ phần phát thử giới hạn biên độ [-1,1] để audio player hoạt động; tín hiệu đọc cho model giữ theo `prepare_audio()`.

In [ ]:
def read_window(row):
    _,_,signal=prepare_audio(row['path'])
    result=signal[int(row['start_sample']):int(row['end_sample'])]
    assert len(result)==WINDOW_SAMPLES and np.isfinite(result).all()
    return result

examples=[]
fig,axes=plt.subplots(len(SPECIES_IDS),2,figsize=(13,10))
for i,species_id in enumerate(SPECIES_IDS):
    options=windows.loc[windows.split.eq('train')&windows.species_id.eq(species_id)]
    assert len(options)>0, 'Không có training window cho một loài.'
    candidates=train_kept.loc[train_kept.species_id.eq(species_id)&train_kept.window_eligible].copy()
    candidates['distance']=(candidates.processed_seconds-candidates.processed_seconds.median()).abs()
    clip=candidates.sort_values(['distance','clip_id']).iloc[0].clip_id
    choices=options.loc[options.clip_id.eq(clip)]
    row=choices.iloc[len(choices)//2]
    signal=read_window(row)
    axes[i,0].plot(np.arange(len(signal))/SAMPLE_RATE,signal,color='#247B91',lw=.7)
    axes[i,0].set_title(f'{row.species} | {row.clip_id}',fontsize=10)
    axes[i,0].set_xlabel('Giây'); axes[i,0].set_ylabel('Biên độ')
    nfft=min(256,WINDOW_SAMPLES); step=min(64,nfft)
    _,_,_,im=axes[i,1].specgram(signal,NFFT=nfft,Fs=SAMPLE_RATE,noverlap=nfft-step,cmap='magma',vmin=-110,vmax=-20)
    axes[i,1].grid(False); axes[i,1].set_ylim(0,SAMPLE_RATE/2)
    axes[i,1].set_xlabel('Giây'); axes[i,1].set_ylabel('Hz'); axes[i,1].set_title(f'{WINDOW_MS} ms sau tiền xử lý',fontsize=10)
    fig.colorbar(im,ax=axes[i,1],label='PSD (dB/Hz)')
    display(Markdown(f'**{row.species}** — `{row.window_id}`'))
    display(Audio(np.clip(signal,-1,1),rate=SAMPLE_RATE,normalize=False))
    examples.append(row.to_dict())
plot_done('05_window_examples')
table(pd.DataFrame(examples),'example_windows')

## 12. Kiểm tra manifest và ranh giới window
Dừng nếu nhóm trùng đã biết đi qua nhiều split, nhóm mâu thuẫn còn giữ, window vượt clip hoặc sai nhãn/split. Đọc window đầu/cuối của bốn ví dụ bằng audio thật.

PASS chỉ xác nhận các điều kiện đã kiểm tra, không chứng minh độc lập nguồn hoặc không có mọi dạng gần trùng.

In [ ]:
checks=[]
def check(name,condition):
    checks.append({'check':name,'passed':bool(condition)})
    assert condition,name
check('Clip manifest không trùng ID',clips.clip_id.is_unique)
check('Giữ nguyên split tham chiếu',clips.reference_split.eq(clips.original_split).all())
check('Không giữ nhóm nhãn mâu thuẫn',not kept.duplicate_group.isin(conflict_groups).any())
check('Mỗi nhóm trùng tối đa một đại diện',kept.duplicate_group.is_unique)
for col in ['sha256','pcm_sha256','canonical_sha256']:
    check(f'{col}: không trùng qua split',kept.groupby(col).split.nunique().le(1).all())
check('Window ID không trùng',windows.window_id.is_unique)
check('Window có đúng số sample',windows.end_sample.sub(windows.start_sample).eq(WINDOW_SAMPLES).all())
check('Start sample không âm và đúng hop',windows.start_sample.ge(0).all() and windows.start_sample.mod(HOP_SAMPLES).eq(0).all())
joined=windows.merge(kept[['clip_id','split','species_id','label_id','processed_frames']],on='clip_id',suffixes=('','_clip'),validate='many_to_one')
check('Window không vượt clip',joined.end_sample.le(joined.processed_frames).all())
check('Nhãn và split window khớp clip',joined.split.eq(joined.split_clip).all() and joined.species_id.eq(joined.species_id_clip).all() and joined.label_id.eq(joined.label_id_clip).all())
check('Tổng window khớp phép tính cấp clip',len(windows)==int(kept.full_windows.sum()))
check('Không tạo window từ clip loại/quá ngắn',set(windows.clip_id)<=set(kept.loc[kept.window_eligible,'clip_id']))
check('Clip bị loại có số window thực xuất bằng 0',clips.loc[clips.status.eq('exclude'),'full_windows'].eq(0).all())
for clip in [row['clip_id'] for row in examples]:
    subset=windows.loc[windows.clip_id.eq(clip)]
    for _,row in subset.iloc[[0,-1]].iterrows(): read_window(row)
check('Đọc window đầu/cuối của bốn ví dụ',True)
exported_clips=pd.read_csv(CLIP_MANIFEST,keep_default_na=False)
exported_windows=pd.read_csv(WINDOW_MANIFEST)
check('CSV clip đọc lại đủ ID',len(exported_clips)==len(clips) and exported_clips.clip_id.is_unique)
check('CSV window đọc lại đủ ID',len(exported_windows)==len(windows) and exported_windows.window_id.is_unique)
check('D5 chỉ bổ sung training Ae. albopictus',
    clips.loc[clips.domain_id.eq(5),'split'].eq('train').all() and clips.loc[clips.domain_id.eq(5),'species_id'].eq(2).all())
check('Validation/test chỉ có domain của bộ chính',kept.loc[kept.split.ne('train'),'domain_id'].isin(DOMAINS).all())
check('Không cắt bớt window lớp lớn',all(
    len(windows.loc[windows.clip_id.eq(row.clip_id)])==row.full_windows for row in kept.itertuples()))
check('Nhóm nguồn/nội dung không giao nhau giữa split',kept.groupby('leakage_group').split.nunique().le(1).all())
check('Không còn name proxy giao nhau trong split nghiên cứu',len(source_warnings)==0)
split_map=kept.set_index('clip_id').split
check('Không có liên kết waveform mạnh giao nhau giữa split',strong_pairs.clip_a.map(split_map).eq(strong_pairs.clip_b.map(split_map)).all())
check('Mỗi split có window của đủ bốn loài',all(set(windows.loc[windows.split.eq(s),'species_id'])==set(SPECIES_IDS) for s in SPLIT_ORDER))
check('Domain test mục tiêu không nằm trong training',not any(row.domain_id==TEST_DOMAINS[row.species_id] for row in kept.loc[kept.split.eq('train')].itertuples()))
check('Pool window không bị cắt ngẫu nhiên cho cân bằng',len(windows)==int(kept.full_windows.sum()))

check('Rà thiết kế đủ bốn loài và khớp domain split thực',
    set(split_design_review.species)==set(ORDER) and all(
        row.train_domains==' | '.join(sorted(kept.loc[kept.species.eq(row.species)&kept.split.eq('train'),'domain'].unique()))
        and row.validation_domains==' | '.join(sorted(kept.loc[kept.species.eq(row.species)&kept.split.eq('validation'),'domain'].unique()))
        and row.test_domains==' | '.join(sorted(kept.loc[kept.species.eq(row.species)&kept.split.eq('test'),'domain'].unique()))
        for row in split_design_review.itertuples()))
check('Bảng nguồn/domain bảo toàn tổng clip và window',
    int(source_domain_coverage.clips.sum())==len(kept) and
    int(source_domain_coverage.windows.sum())==len(windows))
check_results=table(pd.DataFrame(checks),'validation_checks')
display(check_results)

## 13. Report và bản chốt Step A_v1
Report ghi riêng số liệu bản tham chiếu và bản nghiên cứu, phương pháp kiểm tra, quyết định QC, cách chia, metric và giới hạn. `dataset_freeze.csv` lưu phiên bản, thông số và SHA-256 của manifest/bằng chứng để các notebook B–F xác minh cùng một bộ dữ liệu.

Kết luận phạm vi: phần A hoàn tất cho benchmark thăm dò chuyển domain theo từng loài. Dữ liệu sẵn sàng chạy baseline theo protocol A_v1, bảo đảm tách nhóm nguồn đã biết và các đoạn trùng đã phát hiện; chưa phải benchmark bốn loài đã xác minh độc lập cá thể. Không tuyên bố độc lập cá thể hoặc đã tìm hết mọi gần trùng. Validation Ae. albopictus chỉ có một nguồn D1; cần công bố hạn chế này và đánh giá thêm trên dữ liệu mới ở giai đoạn thiết bị.

**Bàn giao B:** kiểm tra SHA-256 trước khi đọc manifest; dùng cột `split`, loại clip không đủ window khỏi metric 300 ms và công bố coverage. Phát triển/chọn model bằng train/validation; chưa dự đoán hoặc đánh giá test trong B–E. Sau khi chốt phương pháp ở F mới đánh giá test. Báo cáo rõ điểm validation Ae. albopictus chỉ phản ánh một nguồn. So sánh baseline theo cùng dữ liệu/ngân sách, thống kê số tham số và kích thước trọng số; chưa kết luận vừa RAM hay chạy được trên ESP32-S3 khi chưa đo thực tế.

In [ ]:
run_time=datetime.now(ZoneInfo('Asia/Ho_Chi_Minh')).isoformat(timespec='seconds')
def frame_html(frame):
    return frame.to_html(index=False,border=0,float_format=lambda x:f'{x:,.4f}',escape=True)
def image_html(name,caption):
    payload=base64.b64encode((FIGURES/f'{name}.png').read_bytes()).decode()
    return f'<figure><img src="data:image/png;base64,{payload}" alt="{html.escape(caption)}"><figcaption>{html.escape(caption)}</figcaption></figure>'
def bullets(items):return '<ul>'+''.join('<li>'+html.escape(x)+'</li>' for x in items)+'</ul>'
split_comparison=reference_counts.stack().rename('reference_clips').to_frame().join(after_counts.stack().rename('research_clips')).reset_index()
split_comparison.columns=['species','split','reference_clips','research_clips']
table(split_comparison,'split_comparison')
policy_rows=[
    ('Protocol',PROTOCOL_VERSION),
    ('Status','Step A complete; exploratory four-species benchmark, not certified independent-individual evaluation'),
    ('Evaluation question','Species classification under class-conditional domain holdout; held-out domain per species, not a globally unseen domain'),
    ('Protocol design review','Keep A_v1; no stronger four-class source-independent in-domain split established in the selected pool under current constraints'),
    ('Species',', '.join(ORDER)),('Base domains',repr(DOMAINS)),
    ('D5',f'Ae. albopictus: {D5_ALBOPICTUS_CLIPS} selected original training clips; seed={SEED}; research training only'),
    ('Frontend preparation',f'mono float32 {SAMPLE_RATE} Hz, window {WINDOW_MS} ms, hop {HOP_MS} ms; no per-clip normalization/augmentation'),
    ('Split', 'reference_split = cleaned original; split = source-proxy/content-group/domain research split'),
    ('Test domains by species ID',repr(TEST_DOMAINS)),
    ('Validation selection',f'seed={SEED}, target {VALIDATION_GROUP_FRACTION:.0%} eligible groups, min {MIN_VALIDATION_ELIGIBLE_CLIPS} eligible clips where feasible; no D5'),
    ('Leakage grouping','matched HumBug name proxies + strongly correlated waveform segments; unknown clips are not verified independent recordings'),
    ('QC decision','Exclude invalid/all-zero and conflicting exact labels; retain finite sparse clipping/low RMS with flags'),
    ('Primary selection metric','Validation clip-level macro-F1: average probabilities across windows per clip, then argmax; short clips excluded from 300 ms metrics and counted separately'),
    ('Final metrics','Clip-level macro-F1, balanced accuracy, per-class recall/confusion, per-domain results; window-level metrics secondary'),
    ('Source metrics','Known-name/group aggregation as supplementary only with coverage reported; no individual-level independence claim'),
    ('B-C comparisons','Natural sampling CE; soft sampling; weighted loss; balanced subset control. Same A_v1 test, no test tuning; compare equivalent training budgets'),
    ('Training-only preprocessing','Any learned normalization, calibration or augmentation fit only on training'),
    ('Test access policy','B-E: train and validation development only; no test prediction, scoring or method selection. F: evaluate after fixing methods and hyperparameters; predeclared runs only'),
    ('Validation limitation','Ae. albopictus validation has one known source; repeated model seeds do not create independent validation sources'),
    ('Edge reporting','Report parameters and weight bytes; floating-point weight size alone does not prove ESP32-S3 runtime/RAM feasibility'),
    ('Sensor scope','D5 and other unmapped clips not certified microphone-only; exploratory signal benchmark, not microphone deployment evidence'),
    ('Limits','Unknown original sources including D5; bounded near-search; Ae. albopictus validation one known D1 source; no sex/pathogen/hardware generalization claim'),
    ('Input metadata version','BioDCASE 2026 Zenodo 20478577 v2; local contents authoritative for sample counts'),
    ('Development ZIP integrity',archive_status+'; MD5='+archive_md5),
    ('Versions',f'Python {platform.python_version()}, numpy {np.__version__}, scipy {scipy.__version__}, soundfile {sf.__version__}')]
protocol_table=table(pd.DataFrame(policy_rows,columns=['item','decision']),'protocol_summary')
metrics_counts=windows.groupby(['species','split']).clip_id.nunique().rename('metric_eligible_clips').reset_index()
table(metrics_counts,'metric_coverage')
findings=[
    f'Chọn {len(clips):,} clip; giữ {len(kept):,}, loại {len(removed):,} bởi quy tắc hợp lệ/trùng/nhãn mâu thuẫn. Không loại thêm chỉ vì RMS thấp.',
    f'Giữ toàn bộ {len(windows):,} chỉ mục window. Đổi split theo nhóm, không tạo thêm tín hiệu hay cắt bớt lớp lớn.',
    f'Tìm {len(strong_pairs)} liên kết dạng sóng mạnh từ {candidate_count:,} cặp ứng viên / {anchor_count:,} anchor. Kết quả chỉ trong phạm vi tìm kiếm đã mô tả.',
    f'Bản tham chiếu có {len(reference_source_warnings)} nhóm name đi qua split; bản A_v1 có {len(source_warnings)}. Nhóm waveform mạnh cũng không giao nhau trong A_v1.',
    'Ae. albopictus: train D5, validation toàn bộ nguồn D1, test D2. Validation có nhiều clip hơn nhưng vẫn chỉ một nguồn đối chiếu; không coi các clip đó là các lần thu độc lập.',
    f'Training A_v1: imbalance lớn nhất/nhỏ nhất {train_ratios["windows"]:.2f} lần theo window; chưa tự chọn sampling/loss.',
    'Nguồn D5/cá thể và loại cảm biến chưa có ánh xạ trong tài liệu và file đã kiểm tra. Không suy ra Wingbeats hoặc microphone cụ thể chỉ từ độ dài 625 ms.',
    'Có clip chạm biên gốc và có overshoot sau resample. Giữ float32 cho tiền xử lý; chỉ playback giới hạn [-1,1], chưa peak-normalize tín hiệu model.',
    'Bản chia nghiên cứu được chốt trước huấn luyện; khác split benchmark chính thức. Bản tham chiếu vẫn có trong manifest để đối chiếu.',
    'Rà thiết kế theo từng loài: giữ A_v1 vì chưa lập được protocol trong cùng domain có đủ nguồn đã xác minh cho cả bốn loài trong pool/ràng buộc hiện tại. Không tạo thêm nguồn bằng cách chia clip/window ngẫu nhiên.',
    'Test là domain giữ riêng theo từng loài, không phải domain hoàn toàn chưa xuất hiện trong training của cả bốn loài. Điểm test đo cả phân biệt loài và chuyển điều kiện thu.',
    'B–E chỉ phát triển bằng train/validation; F mới đánh giá test sau khi chốt phương pháp. Validation albopictus một nguồn cần luôn được công bố, kể cả khi chạy nhiều seed.',
    'Step A_v1 hoàn tất trong phạm vi đã công bố; cần dữ liệu thu mới để kiểm chứng giới tính, cá thể độc lập và chuyển sang INMP441.']
for text in findings:display(Markdown('- '+text))

published_total=218388.40
actual_total=float(audit.seconds.sum())
d5_all=audit.loc[audit.domain_id.eq(5)]
duration_reconciliation=table(pd.DataFrame([
    {'quantity':'All clips: published total seconds','seconds':published_total},
    {'quantity':'All clips: actual header frames/sample_rate','seconds':actual_total},
    {'quantity':'D5: paper table total (265652 x .63)','seconds':167360.76},
    {'quantity':'D5: local WAV total (.625 each)','seconds':float(d5_all.seconds.sum())},
    {'quantity':'Remaining difference after D5 arithmetic','seconds':published_total-actual_total-(167360.76-float(d5_all.seconds.sum()))}],columns=['quantity','seconds']),'duration_reconciliation')
# Kiểm chứng fingerprint cho lần chạy B-F, không dùng file này làm config runtime.
freeze_rows=[{'item':'protocol_version','value':PROTOCOL_VERSION},
    {'item':'stage_status','value':'A complete; B ready for train/validation baseline'},
    {'item':'evaluation_scope','value':'exploratory class-conditional domain holdout; not globally unseen-domain evaluation'},
    {'item':'test_access_policy','value':'No test inference/scoring in B-E; F after fixing methods'},
    {'item':'claim_scope','value':'disjoint known-name proxies and detected waveform groups; unknown individual/session independence not certified'},
    {'item':'sensor_modality_scope','value':'microphone-only not certified for unmapped domains including D5; exploratory benchmark only'},
    {'item':'window_ms','value':WINDOW_MS},{'item':'hop_ms','value':HOP_MS},{'item':'sample_rate','value':SAMPLE_RATE},
    {'item':'seed','value':SEED},{'item':'development_archive_md5','value':archive_md5},{'item':'d5_clip_count','value':D5_ALBOPICTUS_CLIPS},
    {'item':'near_settings','value':repr(near_settings)}]
freeze_files={'clip_manifest':CLIP_MANIFEST,'window_manifest':WINDOW_MANIFEST,
    'input_audio_audit':audit_path,'near_pairs':near_file,'protocol_summary':TABLES/'protocol_summary.csv',
    'selected_d5':TABLES/'selected_d5_clips.csv','quality_decisions':TABLES/'quality_review_decisions.csv',
    'source_domain_coverage':TABLES/'source_domain_coverage.csv',
    'split_design_review':TABLES/'split_design_review.csv','protocol_options':TABLES/'protocol_options.csv'}
for label,file in freeze_files.items():
    freeze_rows.append({'item':label+'_path','value':str(file.relative_to(ROOT))})
    freeze_rows.append({'item':label+'_sha256','value':hashlib.sha256(file.read_bytes()).hexdigest()})
nb_source=nbformat.read(ROOT/'notebooks/A_v1/03_A_data_preparation.ipynb',as_version=4)
freeze_rows.append({'item':'notebook_source_sha256','value':hashlib.sha256('\n'.join(c.source for c in nb_source.cells).encode()).hexdigest()})
freeze_table=table(pd.DataFrame(freeze_rows),'dataset_freeze')
for label,file in freeze_files.items():
    assert hashlib.sha256(file.read_bytes()).hexdigest()==freeze_table.set_index('item').loc[label+'_sha256','value']

sections=[
    '<h1>A_v1 — Dữ liệu bốn loài: nguồn, QC và protocol</h1>',f'<p class="meta">Lần chạy: {html.escape(run_time)}</p>',
    '<div class="notice"><strong>Step A hoàn tất — A_v1 cho baseline thăm dò</strong><p>Không còn giao nhau giữa các nhóm nguồn/nội dung đã biết trong bản nghiên cứu. Đây không phải bảo đảm độc lập cá thể cho nguồn chưa có metadata. Test giữ domain theo từng loài, không phải một domain mới đối với toàn bộ training.</p><p>Ae. albopictus: training D5, validation một nguồn D1, test D2. Chưa xác minh mọi domain là tín hiệu microphone; bản này dùng cho benchmark thăm dò, chưa chứng minh dùng được trên INMP441.</p></div>',
    '<h2>1. Bốn việc đã thực hiện và phạm vi kết luận</h2>',bullets(findings),
    '<h2>2. Nguồn gốc dữ liệu: điều gì đã xác minh</h2>',frame_html(provenance_evidence),frame_html(source_coverage),frame_html(modality_evidence),
    '<p>Không có ánh xạ đầy đủ cá thể/bản ghi gốc trong tài liệu/file đã kiểm tra. Các clip chưa có metadata nguồn không được coi là độc lập chỉ vì ID khác nhau.</p>',
    '<h2>3. Tìm đoạn gần trùng</h2><p>Anchor 300 ms; tối đa 128/clip; bỏ anchor AC RMS dưới 1e-5 và mép 32 ms. Lấy ứng viên theo phổ, xác nhận waveform sau tìm độ trễ. Liên kết mạnh cần tương quan ≥0.9999 ở 300 ms và ≥0.999 trên ít nhất 600 ms, tối đa 1 s. Không xóa bản ghi theo phép thử gần trùng; gộp nhóm khi chia tập. Không phải tìm kiếm vét cạn.</p>',
    frame_html(strong_pairs[['clip_a','clip_b','abs_correlation','extended_abs_correlation','extended_samples']]),
    '<h2>4. Làm sạch và rà chất lượng</h2>',frame_html(removal_summary),frame_html(qc_summary),image_html('04_training_signal_levels','Mức tín hiệu training theo loài; RMS không phải SNR.'),
    '<p>Rà tất cả clip giữ lại bị gắn cờ. Chạm biên trong WAV và overshoot resample được đo riêng. Giữ tín hiệu hữu hạn/nonzero với cờ khi lỗi thưa; không loại chỉ vì RMS thấp. Không tái gán nhãn sinh học bằng kiểm tra tín hiệu.</p>',
    '<p>Bảng đầy đủ kèm lý do ở quality_review_decisions.csv. Không có quyết định tái gán nhãn loài bằng audit này.</p>',frame_html(qc_review[['clip_id','split','native_peak','native_rail_pct','rms_dbfs','decision']]),image_html('06_quality_review','Ví dụ training: waveform và PSD; không tự gán F0/giới tính.'),''.join(qc_audio_html),
    '<h2>5. Chốt cách chia A_v1</h2>',frame_html(protocol_table),image_html('01_before_after_clips','Bản tham chiếu và bản nghiên cứu; thay đổi do chia nhóm, không do cân bằng ngẫu nhiên.'),
    frame_html(research_groups),
    '<h3>Rà tính phù hợp của cách chia</h3>',
    frame_html(split_design_review[['species','train_domains','validation_domains','test_domains','validation_evidence_groups','decision','limitation']]),
    '<p>evidence_groups là nhóm theo bằng chứng hiện có, không phải số cá thể độc lập đã xác minh. Clip không có bằng chứng nguồn vẫn chưa xác minh, dù được cấp nhóm tạm riêng.</p>',
    frame_html(source_domain_coverage),frame_html(protocol_options),
    '<p>Giữ nguyên A_v1 và checksum manifest. Chưa có cơ sở kích hoạt thêm protocol trong cùng domain cho đủ bốn loài theo các ràng buộc đang dùng. Đổi tỷ lệ hoặc seed không khắc phục thiếu nguồn. Kết luận chỉ áp dụng cho pool đã chọn.</p>',
    '<h3>Đối chiếu bản tham chiếu</h3>',frame_html(reference_window_summary),
    '<p>reference_split giữ bản tham chiếu; split là bản nghiên cứu. Test lấy domain chưa thấy theo loài; nếu nhóm chạm test, toàn nhóm vào test. Chọn validation theo nhóm ngoài D5. Không dùng điểm test/model để tạo split.</p>',
    '<h2>6. Window, độ phủ và imbalance</h2>',frame_html(window_summary),frame_html(metrics_counts),
    image_html('02_training_duration_windows','Training theo phút và window, giữ toàn bộ pool hợp lệ.'),image_html('03_training_domain_coverage','Độ phủ domain của training nghiên cứu.'),
    '<p>Window 300 ms/hop 150 ms cắt trong từng clip, không nối/pad. start/end sample tính trên mono 8 kHz, end là cận phải không bao gồm. Clip ngắn vẫn có trong manifest nhưng không được tính metric của cấu hình 300 ms.</p>',
    '<h2>7. Minh họa dữ liệu vào model</h2>',image_html('05_window_examples','Một window training mỗi loài bằng cùng prepare_audio(). STFT chỉ minh họa.'),
    '<h2>8. Kiểm tra và bản chốt</h2>',frame_html(check_results),frame_html(freeze_table),
    '<p>dataset_freeze.csv là bản ghi SHA-256 và thông số, không phải file config runtime. B–F phải dùng đúng protocol và xác minh hash; nếu sửa dữ liệu/split phải tạo phiên bản mới.</p>',
    '<h2>9. Đối chiếu tổng thời lượng</h2>',frame_html(duration_reconciliation),
    '<p>Bảng paper D5 tương ứng 0.63 s/clip, WAV thực là 0.625 s/clip; phép tính này giải thích phần lớn chênh lệch, chưa chứng minh cách tác giả tính/round. Phần dư chưa được giải thích. Pipeline dùng frame/sample_rate thật.</p>',
    '<h2>10. Bước B và giới hạn cần giữ trong báo cáo</h2>',bullets([
        'Chạy baseline trước, so sánh sampling/loss ở B–C trên cùng A_v1; kiểm soát số bước huấn luyện.',
        'B–E không dự đoán/chấm điểm test. Model selection dùng validation clip-level macro-F1; F mới đánh giá test sau khi chốt phương pháp. Tổng hợp xác suất window trong từng clip, không coi overlap là mẫu độc lập.',
        'B dự kiến so sánh đặc trưng phổ + logistic regression với CNN nhỏ trên log-mel; dùng cùng manifest và báo cáo tham số/kích thước trọng số. Phần B cần được duyệt trước khi tạo notebook hoặc thêm thư viện.',
        'Báo cáo clip macro-F1, balanced accuracy, recall từng loài/domain; window metrics là bổ sung. Không suy ra độ chính xác sự kiện bẫy thực tế từ file dài.',
        'Validation Ae. albopictus chỉ một nguồn D1; không có cơ sở cho kết luận ổn định trên mọi nguồn. D5 chưa có cá thể/session ID.',
        'Không tuyên bố nhận diện muỗi cái, mầm bệnh hoặc hoạt động trên INMP441 trước khi có dữ liệu/thử nghiệm phù hợp.']),
    '<p>Nguồn kiểm tra: <a href="https://biodcase.github.io/challenge2026/task5">Task 5</a>, <a href="https://zenodo.org/records/20478577">Zenodo v2</a>, <a href="https://arxiv.org/html/2603.20118v1">baseline paper</a>, metadata local và đối chiếu HumBugDB. Không liên hệ tác giả hoặc gửi dữ liệu ra ngoài.</p>']
css='body{font-family:Segoe UI,Arial,sans-serif;max-width:1160px;margin:36px auto;padding:0 24px;line-height:1.6;color:#243238}h1,h2{line-height:1.3;color:#15576b}h2{margin-top:36px}.notice{padding:18px 22px;background:#fff4df;border-left:5px solid #b87913;margin:24px 0}table{display:block;overflow-x:auto;border-collapse:collapse;width:100%;font-size:13px;margin:18px 0}th,td{padding:8px 10px;border-bottom:1px solid #dbe3e6;text-align:left;vertical-align:top}th{background:#eef5f7}td{overflow-wrap:anywhere}figure{margin:24px 0}img{width:100%;height:auto}figcaption,.meta{font-size:14px;color:#52636a}li{margin-bottom:7px}audio{width:400px;max-width:100%}'
report='<!doctype html><html lang="vi"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1"><title>Step A_v1 — Data audit and protocol</title><style>'+css+'</style></head><body>'+''.join(sections)+'</body></html>'
(OUT/'report.html').write_text(report,encoding='utf-8')
print('Report:',OUT/'report.html')
print('Freeze:',TABLES/'dataset_freeze.csv')
print('Step A_v1 hoàn thành trong phạm vi: không giao nhau giữa nhóm nguồn đã biết và waveform mạnh đã phát hiện.')
